### Overview
For the Bag of Words project, we decided to create a model that used the Complete Works of Shakespeare from Project Gutenberg to predict whether a line of dialogue was spoken by a male or female character. By using some helper functions that we developed, we are able to take the full Shakespeare text and turn it into a CSV file that only contains dialogue along with whether the speaker is male or female. It was a process to clean out all of the act and scene titles as well as all of the play directions. 

After creating the csv file, we were able to tokenize all of the lines and put the tokens into tensors so the model is able to use them properly. We utilized PyTorch's base neural network module to create a Bag of Words model. Using Optuna, we ran hyperparameter optimization on the dimension of the embedding layer, the learning rate, the minimum frequency for words to be tokenized, and the size of the vocabulary.

### Results
After hyperparameter optimization, we trained the model with the training and testing datasets being split up 80/20. The final prediction accuracy we recieved was: 0.617

This is lower than we were hoping for from this model, but there are a couple of things that may have created issues. One is that there are way more male lines than female lines in the dataset, which could skew training and testing results by not weighting female lines enough to counteract the imbalance. Another concern is that many lines are quite simple and do not hold much meaning, which makes predicting the gender of the speaker difficult. Some future improvements we could make include cleaning out the dataset to take out these simple lines with no meaning or combining multiple lines as to create more meaning from them.

Below are 10 testing dataset examples that show the line, the true gender of the speaker, and the prediction our model gave us. As you can see, the model does struggle with getting the correct prediction:

Line: Good gentlemen, give him a further edge,
True: male, Pred: male

Line: What I have said to you.
True: male, Pred: female

Line: And yet my nature never in the fight
True: male, Pred: male

Line: O, I am fortune’s fool!
True: male, Pred: female

Line: To draw with idle spiders’ strings
True: male, Pred: female

Line: But as an honour snatch’d with boisterous hand,
True: male, Pred: male

Line: And in his wisdom, hastes our marriage,
True: male, Pred: female

Line: And if a man did need a poison now,
True: male, Pred: female

Line: That brained my purpose. But peace be with him.
True: male, Pred: female

Line: But why did he swear he would come this morning, and comes not?
True: female, Pred: male

### Creating CSV file from Shakespeare text
Creates list of female and male characters that appear in Shakespeare's works, so that we are able to read the speaker of each line and match them to either male or female. After going through all lines and categorizing them, we write all of the lines to a CSV file. Further cleaning of the text is done to remove all act and scene headings and stage directions.

In [ ]:
import re
import csv

female_chars = {"JULIET", "OPHELIA", "DESDEMONA", "LADY MACBETH", "PORTIA", "VIOLA", "ROSALIND", "BEATRICE", "HERMIONE", "IMOGEN", "CLEOPATRA", "TITANIA", "MIRANDA", "CORDELIA", "REGAN", "GONERIL", "NURSE"}
male_chars = {"ROMEO", "HAMLET", "MACBETH", "OTHELLO", "IAGO", "KING LEAR", "PROSPERO", "BENEDICK", "ORLANDO", "BRUTUS", "CASSIUS", "JULIUS CAESAR", "ANTONY", "HORATIO", "LAERTES", "TYBALT", "MERCUTIO", "BENVOLIO", "FRIAR LAWRENCE", "PARIS", "CAPULET", "MONTAGUE", "DUKE", "KING", "PRINCE"}

def build_dataset(path="shakespeare.txt", out_csv="shakespeare_lines.csv"):
    data = []
    current_speaker = None

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if re.search(r"\b(ACT|SCENE)\b", line):
                continue  # skip act/scene headings

            line = re.sub(r"\[.*?\]", "", line, flags=re.DOTALL)  # remove stage directions in brackets
            
            # detect speaker (all caps + period)
            match = re.match(r"^([A-Z][A-Z\s]+)\.$", line)
            if match:
                speaker = match.group(1).strip()
                current_speaker = speaker
                continue

            if current_speaker in female_chars:
                data.append((line, "female"))
            elif current_speaker in male_chars:
                data.append((line, "male"))

    # write to CSV
    with open(out_csv, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["line", "gender"])
        writer.writerows(data)

    print(f"Saved {len(data)} lines to {out_csv}")

build_dataset("shakespeare.txt", "shakespeare_lines.csv")


Saved 22332 lines to shakespeare_lines.csv


### Importing all Dependencies

In [19]:
import re
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from collections import Counter
import optuna


### Tokenization helper function and Vocabulary class
Defintions of tokenization helper function that takes string and toeknizes each word while removing punctuation and the vocabulary class that takes the word tokens and converts them to integers for the model. ID numbers are also given to the padding token and unknown token. Rare words are also removed based on minimum frequency and the max size of the vocabulary is used to stop creating new tokens.

In [ ]:
def simple_tokenize(s): 
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9\s']", " ", s)
    return [t for t in s.split() if t]

class Vocab: 
    def __init__(self, min_freq=1, max_size=None):
        self.stoi = {"<pad>":0, "<unk>":1}
        self.itos = ["<pad>", "<unk>"]
        self.min_freq = min_freq
        self.max_size = max_size

    def build(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(simple_tokenize(t))
        sorted_words = sorted(counter.items(), key=lambda x: -x[1])
        for word, freq in sorted_words:
            if freq < self.min_freq:
                continue
            if self.max_size and len(self.itos) >= self.max_size:
                break
            if word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def __len__(self):
        return len(self.itos)

    def numericalize(self, tokens):
        return [self.stoi.get(t, 1) for t in tokens]


### LineDataset definition
Using PyTorch's base Dataset class, we created a dataset for our lines which stores the lines in tensors so the model is able to use them properly.

In [ ]:
class LineDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=30):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.vocab.numericalize(simple_tokenize(self.texts[idx]))
        tokens = tokens[:self.max_len] + [0]*(self.max_len-len(tokens))
        return torch.tensor(tokens), torch.tensor(self.labels[idx])

### Bag of Words model class
Using PyTorch's base neural network module, created a Bag of Words classifier model. Model has 2 classes, male and female.

In [ ]:
class CBOWClassifier(nn.Module):  
    def __init__(self, vocab_size, emb_dim=128, num_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, num_classes)

    def forward(self, x):
        emb = self.emb(x)                    
        mask = (x != 0).unsqueeze(-1)      
        avg = (emb * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(avg)


### Prediction helper function
Helper function used for single line evaluations of the model. One line is given and model gives a prediction based on it.

In [ ]:
def predict_line(model, line, vocab, device, max_len=30):
    model.eval()
    tokens = vocab.numericalize(simple_tokenize(line))
    tokens = tokens[:max_len] + [0]*(max_len-len(tokens))
    X = torch.tensor([tokens]).to(device)
    with torch.no_grad():
        out = model(X)
        pred = out.argmax(1).item()
    return "female" if pred == 1 else "male"

### Training and Test texts
CSV file holding all lines is used to create training and testing text datasets. 80% of the lines are used for training, while 20% is using for testing.

In [ ]:
df = pd.read_csv("shakespeare_lines.csv")  
df["label"] = df["gender"].map({"male":0, "female":1})

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["line"].tolist(), df["label"].tolist(), test_size=0.2, random_state=42
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Hyperparamter optimization
Using Optuna, we run 50 trails to optimize the dimension of the embedding layer, the learning rate, the minimum frequency, and the max size of the vocabulary.

In [ ]:
def objective(trial):
    emb_dim = trial.suggest_int("emb_dim", 64, 256)
    lr = trial.suggest_loguniform("lr", 1e-4, 5e-3)
    min_freq = trial.suggest_int("min_freq", 1, 5)
    max_size = trial.suggest_int("max_vocab", 1000, 5000)

    vocab = Vocab(min_freq=min_freq, max_size=max_size)
    vocab.build(train_texts)

    train_ds = LineDataset(train_texts, train_labels, vocab)
    test_ds  = LineDataset(test_texts, test_labels, vocab)
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_dl  = DataLoader(test_ds, batch_size=32)

    model = CBOWClassifier(len(vocab), emb_dim).to(device)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(device))
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(10):
        model.train()
        for X, y in train_dl:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            opt.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best trial:")
print("  Accuracy:", study.best_trial.value)
print("  Params:", study.best_trial.params)


[I 2025-09-28 18:06:32,978] A new study created in memory with name: no-name-1dccec73-a5f5-41c3-be2b-c0f423086548
C:\Users\DSU\AppData\Local\Temp\ipykernel_24288\1596169002.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 5e-3)
[I 2025-09-28 18:07:40,938] Trial 0 finished with value: 0.5997313633310947 and parameters: {'emb_dim': 210, 'lr': 0.0031639618429598563, 'min_freq': 3, 'max_vocab': 3678}. Best is trial 0 with value: 0.5997313633310947.
C:\Users\DSU\AppData\Local\Temp\ipykernel_24288\1596169002.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 5e-3)
[I 2025-09-28 18:08:14,54

Best trial:
  Accuracy: 0.6299529885829416
  Params: {'emb_dim': 242, 'lr': 0.00101629576877008, 'min_freq': 1, 'max_vocab': 4703}


### Final model training
After hyperparameter optimization, we use the best parameter combination to train our model for 100 epochs. The loss function used is cross entropy as it is a classification model.

In [ ]:
best_params = study.best_trial.params
emb_dim = best_params["emb_dim"]
lr = best_params["lr"]
min_freq = best_params["min_freq"]
max_size = best_params["max_vocab"]

vocab = Vocab(min_freq=min_freq, max_size=max_size)
vocab.build(train_texts)

train_ds = LineDataset(train_texts, train_labels, vocab)
test_ds  = LineDataset(test_texts, test_labels, vocab)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=32)

model = CBOWClassifier(len(vocab), emb_dim).to(device)
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).to(device))
opt = torch.optim.Adam(model.parameters(), lr=lr)

for epoch in range(100):
    model.train()
    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        opt.step()

    # Evaluate each epoch
    correct, total = 0, 0
    model.eval()
    with torch.no_grad():
        for X, y in test_dl:
            X, y = X.to(device), y.to(device)
            preds = model(X).argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    print(f"Epoch {epoch+1}: Accuracy={correct/total:.3f}")


Epoch 1: Accuracy=0.499
Epoch 2: Accuracy=0.604
Epoch 3: Accuracy=0.598
Epoch 4: Accuracy=0.580
Epoch 5: Accuracy=0.602
Epoch 6: Accuracy=0.603
Epoch 7: Accuracy=0.617
Epoch 8: Accuracy=0.634
Epoch 9: Accuracy=0.604
Epoch 10: Accuracy=0.604
Epoch 11: Accuracy=0.613
Epoch 12: Accuracy=0.615
Epoch 13: Accuracy=0.622
Epoch 14: Accuracy=0.612
Epoch 15: Accuracy=0.600
Epoch 16: Accuracy=0.607
Epoch 17: Accuracy=0.600
Epoch 18: Accuracy=0.615
Epoch 19: Accuracy=0.609
Epoch 20: Accuracy=0.592
Epoch 21: Accuracy=0.617
Epoch 22: Accuracy=0.607
Epoch 23: Accuracy=0.579
Epoch 24: Accuracy=0.597
Epoch 25: Accuracy=0.612
Epoch 26: Accuracy=0.598
Epoch 27: Accuracy=0.614
Epoch 28: Accuracy=0.598
Epoch 29: Accuracy=0.601
Epoch 30: Accuracy=0.615
Epoch 31: Accuracy=0.625
Epoch 32: Accuracy=0.588
Epoch 33: Accuracy=0.606
Epoch 34: Accuracy=0.599
Epoch 35: Accuracy=0.603
Epoch 36: Accuracy=0.616
Epoch 37: Accuracy=0.608
Epoch 38: Accuracy=0.598
Epoch 39: Accuracy=0.616
Epoch 40: Accuracy=0.605
Epoch 41:

### Single line evaluations
Using the predict_line function defined above, we print out 10 line evaluation examples, where the line is printed out along with the true gender of the speaker and the model's prediction of the speaker's gender.

In [32]:
model.eval()

with torch.no_grad():
    for i in range(10):  # just show 10 examples
        text = test_texts[i]
        true_label = "female" if test_labels[i] == 1 else "male"
        pred = predict_line(model, text, vocab, device)
        print(f"Line: {text}\nTrue: {true_label}, Pred: {pred}\n")


Line: Good gentlemen, give him a further edge,
True: male, Pred: male

Line: What I have said to you.
True: male, Pred: female

Line: And yet my nature never in the fight
True: male, Pred: male

Line: O, I am fortune’s fool!
True: male, Pred: female

Line: To draw with idle spiders’ strings
True: male, Pred: female

Line: But as an honour snatch’d with boisterous hand,
True: male, Pred: male

Line: And in his wisdom, hastes our marriage,
True: male, Pred: female

Line: And if a man did need a poison now,
True: male, Pred: female

Line: That brained my purpose. But peace be with him.
True: male, Pred: female

Line: But why did he swear he would come this morning, and comes not?
True: female, Pred: male

